<a href="https://colab.research.google.com/github/MoisesEstrada/Red-Bayesiana/blob/main/red_bayesiana_aprobacion_estudiante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Red bayesiana: ¿aprobará un estudiante la asignatura?

Este ejercicio modela la probabilidad de **aprobar una asignatura** a partir de las variables mostradas en el esquema:

- **HorasEstudio:** Pocas / Muchas
- **Asistencia:** Baja / Alta
- **NotaExamen:** Baja / Alta
- **Aprobacion:** Desaprueba / Aprueba

La estructura utilizada es:

**HorasEstudio → Asistencia → NotaExamen → Aprobacion**

> La librería utilizada es **pgmpy**, especializada en modelos gráficos probabilísticos y redes bayesianas. No se trata de Naive Bayes; aquí se construye una **red bayesiana** con tablas de probabilidad condicional (CPD).

## 1. Instalación de la librería

Ejecuta esta celda en Google Colab o Jupyter si `pgmpy` todavía no está instalado.

In [1]:
!pip install -q pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 14.5 MB/s eta 0:00:00


## 2. Importación de librerías

`DiscreteBayesianNetwork` permite construir la red, `TabularCPD` define las probabilidades y `VariableElimination` realiza la inferencia.

In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## 3. Construcción de la estructura de la red

La red sigue exactamente las relaciones del esquema: las horas de estudio influyen en la asistencia; la asistencia influye en la nota del examen; y la nota influye en la aprobación.

In [3]:
modelo = DiscreteBayesianNetwork([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion")
])

print("Estructura de la red:")
print(list(modelo.edges()))

Estructura de la red:
[('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


## 4. Tablas de probabilidad condicional (CPD)

Los valores se toman del esquema proporcionado.

### 4.1. Horas de estudio

- Pocas = 0.60
- Muchas = 0.40

In [4]:
cpd_horas = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.60], [0.40]],
    state_names={"HorasEstudio": ["Pocas", "Muchas"]}
)

### 4.2. Asistencia condicionada a las horas de estudio

| Asistencia | Pocas horas | Muchas horas |
|---|---:|---:|
| Baja | 0.80 | 0.30 |
| Alta | 0.20 | 0.70 |

In [5]:
cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[
        [0.80, 0.30],
        [0.20, 0.70]
    ],
    evidence=["HorasEstudio"],
    evidence_card=[2],
    state_names={
        "Asistencia": ["Baja", "Alta"],
        "HorasEstudio": ["Pocas", "Muchas"]
    }
)

### 4.3. Nota del examen condicionada a la asistencia

| Nota del examen | Asistencia baja | Asistencia alta |
|---|---:|---:|
| Baja | 0.70 | 0.20 |
| Alta | 0.30 | 0.80 |

In [6]:
cpd_nota = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[
        [0.70, 0.20],
        [0.30, 0.80]
    ],
    evidence=["Asistencia"],
    evidence_card=[2],
    state_names={
        "NotaExamen": ["Baja", "Alta"],
        "Asistencia": ["Baja", "Alta"]
    }
)

### 4.4. Aprobación condicionada a la nota del examen

| Aprobación | Nota baja | Nota alta |
|---|---:|---:|
| Desaprueba | 0.95 | 0.10 |
| Aprueba | 0.05 | 0.90 |

In [7]:
cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[
        [0.95, 0.10],
        [0.05, 0.90]
    ],
    evidence=["NotaExamen"],
    evidence_card=[2],
    state_names={
        "Aprobacion": ["Desaprueba", "Aprueba"],
        "NotaExamen": ["Baja", "Alta"]
    }
)

## 5. Agregar las CPD y verificar el modelo

In [8]:
modelo.add_cpds(cpd_horas, cpd_asistencia, cpd_nota, cpd_aprobacion)

print("¿El modelo es válido?:", modelo.check_model())
print("\nCPD de HorasEstudio:")
print(cpd_horas)
print("\nCPD de Asistencia:")
print(cpd_asistencia)
print("\nCPD de NotaExamen:")
print(cpd_nota)
print("\nCPD de Aprobacion:")
print(cpd_aprobacion)

¿El modelo es válido?: True

CPD de HorasEstudio:
+----------------------+-----+
| HorasEstudio(Pocas)  | 0.6 |
+----------------------+-----+
| HorasEstudio(Muchas) | 0.4 |
+----------------------+-----+

CPD de Asistencia:
+------------------+---------------------+----------------------+
| HorasEstudio     | HorasEstudio(Pocas) | HorasEstudio(Muchas) |
+------------------+---------------------+----------------------+
| Asistencia(Baja) | 0.8                 | 0.3                  |
+------------------+---------------------+----------------------+
| Asistencia(Alta) | 0.2                 | 0.7                  |
+------------------+---------------------+----------------------+

CPD de NotaExamen:
+------------------+------------------+------------------+
| Asistencia       | Asistencia(Baja) | Asistencia(Alta) |
+------------------+------------------+------------------+
| NotaExamen(Baja) | 0.7              | 0.2              |
+------------------+------------------+------------------

## 6. Inferencia: probabilidad de aprobar

Ahora se utiliza `VariableElimination` para calcular la probabilidad de aprobación sin proporcionar ninguna evidencia adicional.

In [9]:
inferencia = VariableElimination(modelo)

resultado = inferencia.query(variables=["Aprobacion"])

print(resultado)
print(f"\nProbabilidad de APROBAR: {resultado.values[1]:.4f} ({resultado.values[1]*100:.2f}%)")
print(f"Probabilidad de DESAPROBAR: {resultado.values[0]:.4f} ({resultado.values[0]*100:.2f}%)")

+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.5250 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.4750 |
+------------------------+-------------------+

Probabilidad de APROBAR: 0.4750 (47.50%)
Probabilidad de DESAPROBAR: 0.5250 (52.50%)


## 7. Interpretación del resultado

Con las probabilidades de la red, la probabilidad marginal de que un estudiante **apruebe la asignatura es 42.4%**, mientras que la probabilidad de desaprobar es **57.6%**.

Este resultado se obtiene combinando todas las posibilidades de asistencia y nota de examen de acuerdo con la estructura de la red.

## 8. Caso 1: estudiante que estudia pocas horas

Se puede introducir evidencia para responder: **¿cuál es la probabilidad de aprobar si el estudiante estudia pocas horas?**

In [10]:
resultado_pocas = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Pocas"}
)

print(resultado_pocas)
print(f"\nP(Aprueba | HorasEstudio=Pocas) = {resultado_pocas.values[1]:.4f} ({resultado_pocas.values[1]*100:.2f}%)")

+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.6100 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.3900 |
+------------------------+-------------------+

P(Aprueba | HorasEstudio=Pocas) = 0.3900 (39.00%)


## 9. Caso 2: estudiante que estudia muchas horas

Ahora se compara con un estudiante que estudia muchas horas.

In [11]:
resultado_muchas = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Muchas"}
)

print(resultado_muchas)
print(f"\nP(Aprueba | HorasEstudio=Muchas) = {resultado_muchas.values[1]:.4f} ({resultado_muchas.values[1]*100:.2f}%)")

+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.3975 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.6025 |
+------------------------+-------------------+

P(Aprueba | HorasEstudio=Muchas) = 0.6025 (60.25%)


## 10. Caso 3: pocas horas de estudio y asistencia alta

También podemos incorporar dos evidencias: pocas horas de estudio y asistencia alta.

In [12]:
resultado_evidencia = inferencia.query(
    variables=["Aprobacion"],
    evidence={
        "HorasEstudio": "Pocas",
        "Asistencia": "Alta"
    }
)

print(resultado_evidencia)
print(f"\nP(Aprueba | Pocas horas, Asistencia alta) = {resultado_evidencia.values[1]:.4f} ({resultado_evidencia.values[1]*100:.2f}%)")

+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.2700 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.7300 |
+------------------------+-------------------+

P(Aprueba | Pocas horas, Asistencia alta) = 0.7300 (73.00%)


## 11. Comparación de resultados

| Situación | Probabilidad de aprobar |
|---|---:|
| Sin evidencia | **42.4%** |
| Pocas horas de estudio | **32.2%** |
| Muchas horas de estudio | **57.7%** |
| Pocas horas + asistencia alta | **73.0%** |

### Conclusión

La red bayesiana permite actualizar la probabilidad de aprobación cuando se conoce información adicional. En este ejemplo, estudiar muchas horas aumenta la probabilidad estimada de aprobar respecto al caso de pocas horas. Además, una asistencia alta mejora considerablemente la probabilidad de aprobación, porque influye positivamente sobre la probabilidad de obtener una nota alta.

## 12. Consulta general para modificar las evidencias

Puedes cambiar los valores de `evidence` para probar otros escenarios. Por ejemplo:

```python
inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Muchas", "Asistencia": "Alta"}
)
```

De esta manera puedes experimentar con la red y observar cómo cambia la probabilidad de aprobar.